# 🚀 Urban Mission Planning - GPU Training on Google Colab

This notebook trains your road segmentation model with GPU acceleration.

**Before you start:**
1. Enable GPU: `Runtime` → `Change runtime type` → `Hardware accelerator: GPU`
2. Upload your data to Google Drive: `MyDrive/ump_data/reference/sats/` and `MyDrive/ump_data/reference/maps/`
3. This notebook will automatically clone your code from GitHub

## ✅ Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected!")
    print("Go to: Runtime → Change runtime type → Hardware accelerator: GPU")

## 📦 Step 2: Install Dependencies

In [ ]:
%%capture
!pip install segmentation-models-pytorch
!pip install opencv-python-headless
!pip install scikit-image
!pip install albumentations

print("✓ All dependencies installed")

## 📥 Step 3: Clone Project from GitHub

In [ ]:
import os

# Clone repository
print("📥 Cloning repository from GitHub...")
!git clone https://github.com/Shreyy8/PathFinding_MLModel.git /content/project

# Change to project directory
%cd /content/project

# Verify structure
print("\n📂 Project structure:")
!ls -la

if os.path.exists('src'):
    print("\n📂 Source files:")
    !ls -la src/
    print("\n✓ Project cloned successfully!")
else:
    print("\n⚠️ WARNING: 'src' folder not found!")

## 💾 Step 4: Mount Google Drive (for training data)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

## 🔍 Step 5: Verify Training Data

In [ ]:
import os

# Data paths
IMAGES_DIR = "/content/drive/MyDrive/ump_data/reference/sats"
MASKS_DIR = "/content/drive/MyDrive/ump_data/reference/maps"

# Check if directories exist
if os.path.exists(IMAGES_DIR) and os.path.exists(MASKS_DIR):
    num_images = len([f for f in os.listdir(IMAGES_DIR) if f.endswith('.tif') or f.endswith('.tiff')])
    num_masks = len([f for f in os.listdir(MASKS_DIR) if f.endswith('.tif') or f.endswith('.tiff')])
    
    print(f"✓ Found {num_images} satellite images")
    print(f"✓ Found {num_masks} ground truth masks")
    
    if num_images == 0 or num_masks == 0:
        print("\n⚠️ WARNING: No .tif/.tiff files found!")
        print("Please upload your training data to Google Drive.")
    elif num_images != num_masks:
        print(f"\n⚠️ WARNING: Mismatch between images ({num_images}) and masks ({num_masks})")
    else:
        print("\n✓ Data verification passed!")
        print(f"\nSample files:")
        !ls {IMAGES_DIR} | head -5
else:
    print("❌ ERROR: Data directories not found!")
    print(f"\nExpected paths:")
    print(f"  Images: {IMAGES_DIR}")
    print(f"  Masks: {MASKS_DIR}")
    print("\nPlease upload your data to these locations in Google Drive.")

## 🔧 Step 6: Create Training Script

In [ ]:
%%writefile /content/project/train_colab.py
import torch
import numpy as np
import logging
from pathlib import Path
from tqdm import tqdm
import sys
import os

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import project modules
from src.road_segmentation_model import RoadSegmentationModel
from src.dataset import create_dataloaders
from src.loss_functions import create_loss_function

def compute_iou(predictions, targets, threshold=0.5):
    predictions_binary = (torch.sigmoid(predictions) > threshold).float()
    targets = targets.float()
    
    predictions_flat = predictions_binary.view(-1)
    targets_flat = targets.view(-1)
    
    intersection = (predictions_flat * targets_flat).sum()
    union = predictions_flat.sum() + targets_flat.sum() - intersection
    
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    
    return (intersection / union).item()

def evaluate_iou(model, dataloader, device):
    model.model.eval()
    total_iou = 0.0
    num_batches = 0
    
    logger.info('Evaluating IoU on validation set...')
    
    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc='Computing IoU'):
            images = images.to(device)
            masks = masks.to(device)
            
            predictions = model.model(images)
            batch_iou = compute_iou(predictions, masks)
            total_iou += batch_iou
            num_batches += 1
    
    avg_iou = total_iou / num_batches
    return avg_iou

def train_model(images_dir, masks_dir, num_epochs=15, encoder='resnet18'):
    logger.info('='*80)
    logger.info('🚀 Training on Google Colab with GPU')
    logger.info('='*80)
    
    BATCH_SIZE = 4
    LEARNING_RATE = 0.001
    
    logger.info(f'Configuration:')
    logger.info(f'  Epochs: {num_epochs}')
    logger.info(f'  Encoder: {encoder}')
    logger.info(f'  Batch size: {BATCH_SIZE}')
    logger.info(f'  Learning rate: {LEARNING_RATE}')
    
    logger.info('\n📊 Creating dataloaders...')
    train_loader, val_loader = create_dataloaders(
        images_dir=images_dir,
        masks_dir=masks_dir,
        batch_size=BATCH_SIZE,
        train_split=0.8,
        num_workers=2,
        augment_train=True,
        seed=42
    )
    logger.info(f'✓ Training samples: {len(train_loader.dataset)}')
    logger.info(f'✓ Validation samples: {len(val_loader.dataset)}')
    
    logger.info('\n🤖 Initializing model...')
    model = RoadSegmentationModel(
        architecture='unet',
        encoder_name=encoder,
        encoder_weights='imagenet',
        in_channels=3,
        out_classes=1
    )
    
    num_params = sum(p.numel() for p in model.model.parameters())
    logger.info(f'✓ Device: {model.device}')
    logger.info(f'✓ Parameters: {num_params:,}')
    
    logger.info('\n📉 Creating loss function...')
    loss_function = create_loss_function('dice')
    logger.info('✓ Using Dice loss')
    
    os.makedirs('/content/models', exist_ok=True)
    
    logger.info(f'\n🏋️ Starting training for {num_epochs} epochs...')
    logger.info('='*80)
    
    history = model.train_model(
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=num_epochs,
        loss_function=loss_function,
        optimizer_type='adam',
        learning_rate=LEARNING_RATE,
        checkpoint_dir='/content/models',
        checkpoint_name='trained_model.pth'
    )
    
    logger.info('\n📊 Evaluating final IoU...')
    checkpoint_path = Path('/content/models/trained_model.pth')
    model.load_checkpoint(str(checkpoint_path))
    val_iou = evaluate_iou(model, val_loader, model.device)
    
    logger.info('\n' + '='*80)
    logger.info('🎉 Training Complete!')
    logger.info('='*80)
    logger.info(f'Best epoch: {history[\"best_epoch\"]}')
    logger.info(f'Best validation loss: {history[\"best_val_loss\"]:.4f}')
    logger.info(f'Final training loss: {history[\"train_loss\"][-1]:.4f}')
    logger.info(f'Final validation loss: {history[\"val_loss\"][-1]:.4f}')
    logger.info(f'Validation IoU: {val_iou:.4f}')
    logger.info(f'Model saved to: {checkpoint_path}')
    logger.info('='*80)
    
    if val_iou >= 0.85:
        logger.info(f'\n✅ SUCCESS: IoU {val_iou:.4f} >= 0.85 (Requirement met!)')
    elif val_iou >= 0.75:
        logger.info(f'\n⚠️ GOOD: IoU {val_iou:.4f} is decent for {num_epochs} epochs')
        logger.info('   Consider training for more epochs to reach 0.85')
    else:
        logger.info(f'\n⚠️ LOW: IoU {val_iou:.4f} - may need more training')
    
    return val_iou, history

if __name__ == '__main__':
    import argparse
    
    parser = argparse.ArgumentParser()
    parser.add_argument('--images', required=True, help='Path to satellite images')
    parser.add_argument('--masks', required=True, help='Path to ground truth masks')
    parser.add_argument('--epochs', type=int, default=15, help='Number of epochs')
    parser.add_argument('--encoder', default='resnet18', help='Encoder architecture')
    
    args = parser.parse_args()
    
    val_iou, history = train_model(
        images_dir=args.images,
        masks_dir=args.masks,
        num_epochs=args.epochs,
        encoder=args.encoder
    )
    
    print(f'\n🎯 Final IoU: {val_iou:.4f}')

print('✓ Training script created')

## 🏃 Step 7: Train Model (Lightweight - 15 epochs)

This will train for 15 epochs with ResNet18 encoder (~10-15 minutes on GPU)

In [ ]:
# Train with lightweight settings
!python train_colab.py \
    --images "/content/drive/MyDrive/ump_data/reference/sats" \
    --masks "/content/drive/MyDrive/ump_data/reference/maps" \
    --epochs 15 \
    --encoder resnet18

## 🚀 Step 8 (Optional): Full Training (50 epochs)

Run this cell if you want better accuracy. Uses ResNet34 encoder (~30-45 minutes on GPU)

In [ ]:
# Uncomment to run full training
# !python train_colab.py \
#     --images "/content/drive/MyDrive/ump_data/reference/sats" \
#     --masks "/content/drive/MyDrive/ump_data/reference/maps" \
#     --epochs 50 \
#     --encoder resnet34

## 💾 Step 9: Download Trained Model

In [ ]:
from google.colab import files
import os

model_path = "/content/models/trained_model.pth"

if os.path.exists(model_path):
    # Download to your computer
    files.download(model_path)
    print("✓ Model downloaded to your computer")
    
    # Also backup to Google Drive
    !cp /content/models/trained_model.pth /content/drive/MyDrive/
    print("✓ Model backed up to Google Drive")
    
    # Show file size
    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"\nModel size: {size_mb:.2f} MB")
else:
    print("❌ Model file not found. Training may have failed.")

## 🧪 Step 10: Test the Model (Optional)

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from src.road_segmentation_model import RoadSegmentationModel
import os

# Load model
print("Loading model...")
model = RoadSegmentationModel(
    architecture="unet",
    encoder_name="resnet18",
    encoder_weights=None
)
model.load_checkpoint("/content/models/trained_model.pth")
print("✓ Model loaded")

# Load a test image
images_dir = "/content/drive/MyDrive/ump_data/reference/sats"
test_images = [f for f in os.listdir(images_dir) if f.endswith('.tif') or f.endswith('.tiff')]

if test_images:
    # Load first image
    test_img_path = os.path.join(images_dir, test_images[0])
    print(f"Testing on: {test_images[0]}")
    
    image = Image.open(test_img_path)
    image_array = np.array(image)
    
    # Predict
    print("Generating prediction...")
    mask = model.predict(image_array)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    
    axes[0].imshow(image_array)
    axes[0].set_title("Input Satellite Image", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Predicted Road Mask", fontsize=14)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Calculate road coverage
    road_pixels = np.sum(mask > 0)
    total_pixels = mask.size
    coverage = (road_pixels / total_pixels) * 100
    
    print(f"\n✓ Prediction successful!")
    print(f"Road coverage: {coverage:.2f}%")
else:
    print("No test images found")

## 📋 Summary

**What you accomplished:**
- ✅ Cloned project from GitHub
- ✅ Trained road segmentation model on GPU
- ✅ Achieved IoU score on validation set
- ✅ Downloaded trained model
- ✅ Backed up model to Google Drive

**Next steps:**
1. Copy `trained_model.pth` to your local `models/` folder
2. Rename it to `best_model.pth` if needed
3. Use it for inference on test images

**Files created:**
- `/content/models/trained_model.pth` - Your trained model
- `MyDrive/trained_model.pth` - Backup in Google Drive

## 🗺️ Step 11: Interactive Pathfinding

Find the shortest path between any two points on the road network!

In [ ]:
# Interactive pathfinding - Choose your image and coordinates
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
from src.solution_generator import SolutionGenerator
from src.road_segmentation_model import RoadSegmentationModel
from src.image_preprocessor import ImagePreprocessor

# ========== CONFIGURATION - CHANGE THESE ==========
# Choose your image (change the filename)
IMAGE_NAME = "train_001.tiff"  # Options: train_001.tiff, train_002.tiff, etc.

# Choose start and goal coordinates (x, y)
START_POINT = (57, 0)      # Change these
GOAL_POINT = (532, 1499)   # Change these
# ==================================================

# Build full path
images_dir = "/content/drive/MyDrive/ump_data/reference/sats"
image_path = os.path.join(images_dir, IMAGE_NAME)

print(f"Selected image: {IMAGE_NAME}")
print(f"Start: {START_POINT}, Goal: {GOAL_POINT}")

# Check if image exists
if not os.path.exists(image_path):
    print(f"\n❌ Error: Image not found at {image_path}")
    print("\nAvailable images:")
    for f in sorted(os.listdir(images_dir))[:10]:
        if f.endswith('.tiff') or f.endswith('.tif'):
            print(f"  - {f}")
else:
    # Load image
    image = Image.open(image_path)
    image_array = np.array(image)
    
    # Load model (only once)
    if 'pathfinding_model' not in globals():
        print("\nLoading model...")
        pathfinding_model = RoadSegmentationModel(
            architecture="unet",
            encoder_name="resnet34",
            encoder_weights=None
        )
        pathfinding_model.load_checkpoint("/content/project/models/best_model.pth")
        print("✓ Model loaded")
    
    # First, visualize roads to help pick coordinates
    print("\n📊 Analyzing roads...")
    preprocessor = ImagePreprocessor(min_size=1500, max_size=8192)
    image_tensor = preprocessor.preprocess(image_path)
    road_mask = pathfinding_model.predict(image_tensor)
    
    road_pixels = np.argwhere(road_mask > 0)
    print(f"Road coverage: {np.mean(road_mask)*100:.2f}%")
    print(f"Road pixels: {len(road_pixels)}")
    
    # Show sample valid coordinates
    if len(road_pixels) > 0:
        print("\n✓ Sample valid road coordinates:")
        indices = [0, len(road_pixels)//4, len(road_pixels)//2, 3*len(road_pixels)//4, -1]
        for i, idx in enumerate(indices):
            y, x = road_pixels[idx]
            print(f"  Option {i+1}: ({x}, {y})")
    
    # Create solution generator
    config = {'MIN_IMAGE_SIZE': 1500, 'MAX_IMAGE_SIZE': 8192}
    solution_gen = SolutionGenerator(model=pathfinding_model, config=config)
    
    # Find path
    print(f"\n🔍 Finding shortest path...")
    try:
        result = solution_gen.process_image(
            image_path=image_path,
            start=START_POINT,
            goal=GOAL_POINT
        )
        
        # Display results
        print(f"\n✓ Path found!")
        print(f"  Waypoints: {len(result['path'])}")
        print(f"  Path length: {result['validation']['path_length']:.2f} pixels")
        print(f"  Violations: {result['validation']['violations']}")
        print(f"  Score: {result['validation']['score']:.2f}")
        
        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(20, 7))
        
        # Original image
        axes[0].imshow(image_array)
        axes[0].set_title(f"{IMAGE_NAME}", fontsize=14)
        axes[0].axis('off')
        
        # Road overlay
        axes[1].imshow(image_array)
        axes[1].imshow(road_mask, cmap='Reds', alpha=0.5)
        axes[1].set_title("Roads (red areas)", fontsize=14)
        axes[1].axis('off')
        
        # Path
        axes[2].imshow(image_array)
        path_array = np.array(result['path'])
        axes[2].plot(path_array[:, 0], path_array[:, 1], 'r-', linewidth=2, label='Path')
        axes[2].scatter(path_array[:, 0], path_array[:, 1], c='yellow', s=20, zorder=5)
        axes[2].scatter(START_POINT[0], START_POINT[1], c='green', s=200, marker='o', zorder=10, label='Start')
        axes[2].scatter(GOAL_POINT[0], GOAL_POINT[1], c='red', s=200, marker='X', zorder=10, label='Goal')
        axes[2].set_title(f"Score: {result['validation']['score']:.0f}", fontsize=14)
        axes[2].legend()
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Show path coordinates
        print(f"\nPath waypoints (first 10):")
        for i, point in enumerate(result['path'][:10]):
            print(f"  {i+1}. {point}")
        if len(result['path']) > 10:
            print(f"  ... and {len(result['path']) - 10} more")
            
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("\n💡 Tip: Make sure START_POINT and GOAL_POINT are on roads (red areas)")
        print("Use the 'Sample valid road coordinates' shown above!")